# Segmentation Benchmark — full Colab run

This notebook runs the entire benchmark end-to-end on Colab's GPU. Expected runtime on a T4:
- Data preparation: 15–25 minutes (COCO 2017 downloads are ~20 GB)
- Full training of all 10 models at 512×512 for 25 epochs on 5000 training images: ~10–14 hours
- With `--smoke` (200 train / 40 val / 40 test, 3 epochs): ~40 minutes total

If you are time-constrained, run the smoke test first to verify the pipeline works, then start the full run and let it complete in the background.


## 1. Attach a GPU
**Runtime → Change runtime type → T4 GPU** (or better).

In [1]:
!nvidia-smi

Mon Sep 21 18:52:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone the repository

In [4]:
# From your GitHub after you push, or use a local upload if not yet published
%cd /content
!git clone https://github.com/savannahshannon/savannah_shannon_segmentation_benchmark.git
%cd /content/savannah_shannon_segmentation_benchmark

/content
Cloning into 'savannah_shannon_segmentation_benchmark'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 74 (delta 12), reused 70 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 999.89 KiB | 24.39 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/savannah_shannon_segmentation_benchmark


## 3. Install dependencies

In [5]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 8.6 MB/s eta 0:00:00


## 4. Prepare the COCO 2017 subset (SEED=42)

In [21]:
!rm -rf results/kmeans results/fcn results/unet results/unetpp results/segnet \
        results/deeplabv3 results/pspnet results/segformer results/maskrcnn results/yolo_seg
!rm -f  results/*.csv
!rm -rf predictions/* confusion_matrices/* logs/* checkpoints/*
!echo "cleared"

cleared


In [11]:
!python scripts/prepare_coco_subset.py --train 300 --val 60 --test 60 --yolo-yaml

downloading http://images.cocodataset.org/zips/train2017.zip -> coco2017/_cache/train2017.zip
downloading http://images.cocodataset.org/zips/val2017.zip -> coco2017/_cache/val2017.zip
subset built: train=300, val=60, test=60
YOLO dataset YAML: annotations/yolo_seg.yaml


In [24]:
# For a smoke test use small numbers first to verify the pipeline works end-to-end.
# Full protocol (assignment §5): 5000/1000/1000.
!python scripts/prepare_coco_subset.py --train 5000 --val 1000 --test 1000 --yolo-yaml

subset built: train=5000, val=1000, test=1000
YOLO dataset YAML: annotations/yolo_seg.yaml


## 5. (Optional) Quick smoke run — verifies every model works before the full run

In [16]:
!python run_benchmark.py --task all --model all --smoke


############ deeplabv3 (semantic) ############
=== training deeplabv3 on cuda ===
/content/savannah_shannon_segmentation_benchmark/src/train.py:63: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(mixed_precision and device.startswith("cuda")))
/content/savannah_shannon_segmentation_benchmark/src/train.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(mixed_precision and device.startswith("cuda"))):
[epoch 001] train_loss=0.9395 val_loss=0.6747 miou=0.3466 dice=0.4006 pa=0.9186 time=35.2s
[epoch 002] train_loss=0.6822 val_loss=0.5946 miou=0.5708 dice=0.6476 pa=0.9436 time=37.7s
[epoch 003] train_loss=0.5985 val_loss=0.5595 miou=0.5910 dice=0.6636 pa=0.9477 time=40.2s
=== evaluating deeplabv3 on test set ===

############ fcn (semantic) #########

## 6. Full benchmark run

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
!python run_benchmark.py --task all --model all 2>&1 | tee logs/full_run.log


############ kmeans (traditional) ############
kmeans: 100%|██████████| 125/125 [02:05<00:00,  1.00s/it]
/content/savannah_shannon_segmentation_benchmark/src/train.py:63: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(mixed_precision and device.startswith("cuda")))

############ fcn (semantic) ############
=== training fcn on cuda ===
/content/savannah_shannon_segmentation_benchmark/src/train.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(mixed_precision and device.startswith("cuda"))):
[epoch 001] train_loss=0.4783 val_loss=0.3906 miou=0.5123 dice=0.6424 pa=0.9385 time=474.3s
[epoch 002] train_loss=0.3853 val_loss=0.4127 miou=0.5199 dice=0.6570 pa=0.9229 time=477.4s
[epoch 003] train_loss=0.3684 val_loss=0.3778 miou=0.5430 dice=0.6799 pa=0

In [25]:
!python run_benchmark.py --task all --model all


############ kmeans (traditional) ############
Traceback (most recent call last):
  File "/content/savannah_shannon_segmentation_benchmark/run_benchmark.py", line 65, in <module>
    raise SystemExit(main())
                     ~~~~^^
  File "/content/savannah_shannon_segmentation_benchmark/run_benchmark.py", line 60, in main
    run_benchmark(args.config, task=args.task, model=args.model, out_root=args.out)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/savannah_shannon_segmentation_benchmark/src/benchmark.py", line 454, in run_benchmark
    run_traditional_kmeans(cfg, out_root)
    ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
  File "/content/savannah_shannon_segmentation_benchmark/src/benchmark.py", line 176, in run_traditional_kmeans
    _, _, test_loader = build_loaders(cfg, task="semantic", num_workers=0)
                        ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/savannah_shannon_segmentation_benchm

## 7. Regenerate the aggregated CSVs and plots (fast, no training)

In [ ]:
!python run_benchmark.py --aggregate-only
import pandas as pd
pd.read_csv('results/semantic_segmentation_results.csv')

## 8. Inspect a few qualitative comparisons

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os
for m in ['unet', 'deeplabv3', 'segformer']:
    files = sorted(os.listdir(f'predictions/{m}'))[:3]
    fig, axes = plt.subplots(1, len(files), figsize=(12, 4))
    for ax, f in zip(axes, files):
        ax.imshow(Image.open(f'predictions/{m}/{f}'))
        ax.set_title(f'{m}/{f}'); ax.axis('off')
    plt.show()

## 9. Persist results back to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/Segmentation_Benchmark_Results
!cp -r results plots predictions logs confusion_matrices /content/drive/MyDrive/Segmentation_Benchmark_Results/
print('Results copied to Google Drive.')